# 사용자 개인화 추천 시스템 (하이브리드 필터링)

사용자별 개인화 추천을 생성합니다.
- **협업 필터링 (CF)**: 유사 유저의 평가 기반 추천
- **콘텐츠 기반 필터링 (CBF)**: 유저 프로필 벡터 ↔ 애니 특징 벡터 유사도
- **콜드 스타트**: 장르 선호도 기반 CBF 프로필 생성
- **하이브리드**: α × CF + (1-α) × CBF (상호작용 수에 따라 α 조정)

결과를 `user_recommendations` 테이블에 유저별 Top-20으로 저장합니다.

In [27]:
import dotenv
import os
import json
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
dotenv.load_dotenv(dotenv.find_dotenv())

pg = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)
cur = pg.cursor()
print("DB 연결 완료")

DB 연결 완료


## 데이터 수집

In [29]:
# 1) 명시적 피드백: anime_reviews (user_id 또는 external_user_id, anime_id, score)
#    - 사이트 유저: user_id 존재, external_user_id NULL
#    - 크롤링 유저: user_id NULL, external_user_id 존재
cur.execute("""
    SELECT user_id, external_user_id, anime_id, score
    FROM anime_reviews
    WHERE deleted_at IS NULL
""")
reviews_raw = cur.fetchall()
print(f"리뷰 수: {len(reviews_raw)}")

site_review_count = sum(1 for r in reviews_raw if r[0] is not None)
ext_review_count = sum(1 for r in reviews_raw if r[1] is not None)
print(f"  사이트 유저 리뷰: {site_review_count}, 크롤링 유저 리뷰: {ext_review_count}")

# 2) 암묵적 피드백: anime_watches (user_id, anime_id) → 사이트 유저만
cur.execute("""
    SELECT user_id, anime_id
    FROM anime_watches
""")
watches = cur.fetchall()
print(f"시청 기록 수: {len(watches)}")

# 3) 애니메이션 메타데이터
cur.execute("""
    SELECT a.id, m.genres, m.tags, m.directors, m.production_companies
    FROM anime a
    LEFT JOIN anime_metadata m ON a.id = m.anime_id
    ORDER BY a.id
""")
metadata_rows = cur.fetchall()
print(f"애니메이션 수: {len(metadata_rows)}")

# 4) 유저 장르 선호도 (사이트 유저만)
cur.execute("""
    SELECT user_id, genre FROM user_genre_preferences
""")
genre_prefs = cur.fetchall()
print(f"장르 선호도 레코드 수: {len(genre_prefs)}")

# 5) series_mapping: anime_id → group_name (동일 시리즈 중복 제거용)
cur.execute("SELECT anime_id, group_name FROM series_mapping")
series_rows = cur.fetchall()
anime_group_map = {aid: gname for aid, gname in series_rows}
print(f"시리즈 매핑 수: {len(anime_group_map)}")

리뷰 수: 672878
  사이트 유저 리뷰: 6, 크롤링 유저 리뷰: 672872
시청 기록 수: 6
애니메이션 수: 8873
장르 선호도 레코드 수: 0
시리즈 매핑 수: 8873


In [30]:
# 애니메이션 메타데이터 파싱
anime_ids = []
genres_list = []
tags_list = []
directors_list = []
companies_list = []

for row in metadata_rows:
    aid, genres_json, tags_json, directors_json, companies_json = row
    anime_ids.append(aid)

    genres = genres_json if isinstance(genres_json, list) else (json.loads(genres_json) if genres_json else [])
    genres_list.append(genres)

    tags = tags_json if isinstance(tags_json, list) else (json.loads(tags_json) if tags_json else [])
    tags_list.append(tags)

    directors_raw = directors_json if isinstance(directors_json, list) else (json.loads(directors_json) if directors_json else [])
    directors_list.append([d["name"] for d in directors_raw if isinstance(d, dict) and "name" in d])

    companies_raw = companies_json if isinstance(companies_json, list) else (json.loads(companies_json) if companies_json else [])
    companies_list.append([c["name"] for c in companies_raw if isinstance(c, dict) and "name" in c])

anime_id_to_idx = {aid: i for i, aid in enumerate(anime_ids)}
print(f"Parsed {len(anime_ids)} anime")

# ── 통합 유저 키 구성 ──
# 사이트 유저: ("site", user_id) / 크롤링 유저: ("ext", external_user_id)
# CF 행렬에는 모두 포함, 추천 결과는 사이트 유저에게만 생성
all_user_keys = set()
site_user_keys = set()

# 리뷰 → (unified_key, anime_id, score)로 변환
reviews = []
for user_id, ext_user_id, anime_id, score in reviews_raw:
    if user_id is not None:
        key = ("site", user_id)
        site_user_keys.add(key)
    elif ext_user_id is not None:
        key = ("ext", ext_user_id)
    else:
        continue
    all_user_keys.add(key)
    reviews.append((key, anime_id, score))

for uid, _ in watches:
    key = ("site", uid)
    all_user_keys.add(key)
    site_user_keys.add(key)

for uid, _ in genre_prefs:
    key = ("site", uid)
    all_user_keys.add(key)
    site_user_keys.add(key)

# 정렬: 사이트 유저 먼저, 크롤링 유저 나중에
user_keys = sorted(all_user_keys, key=lambda k: (0 if k[0] == "site" else 1, k[1]))
user_key_to_idx = {k: i for i, k in enumerate(user_keys)}

n_site_users = len(site_user_keys)
n_all_users = len(user_keys)
print(f"사이트 유저 수: {n_site_users}, 크롤링 유저 수: {n_all_users - n_site_users}, 전체: {n_all_users}")

# 사이트 유저 인덱스 → 실제 user_id 매핑
site_idx_to_user_id = {}
for key in site_user_keys:
    site_idx_to_user_id[user_key_to_idx[key]] = key[1]

site_indices = sorted(site_idx_to_user_id.keys())

# 유저별 장르 선호도 매핑
user_genre_map = {}
for uid, genre in genre_prefs:
    key = ("site", uid)
    user_genre_map.setdefault(key, []).append(genre)

Parsed 8873 anime
사이트 유저 수: 2, 크롤링 유저 수: 184935, 전체: 184937


## User-Anime 상호작용 행렬 (Sparse Matrix)

In [31]:
# User-Anime 상호작용 행렬 구성 (전체 유저: 사이트 + 크롤링)
# 리뷰 score 우선, 시청만 한 경우 암묵적 score = 6.0
n_anime = len(anime_ids)

interaction = {}  # (user_idx, anime_idx) → score

# 리뷰 (명시적)
for key, aid, score in reviews:
    if key in user_key_to_idx and aid in anime_id_to_idx:
        interaction[(user_key_to_idx[key], anime_id_to_idx[aid])] = float(score)

# 시청 (암묵적, 리뷰 없는 경우만) — 사이트 유저만
IMPLICIT_SCORE = 6.0
for uid, aid in watches:
    key = ("site", uid)
    if key in user_key_to_idx and aid in anime_id_to_idx:
        pair = (user_key_to_idx[key], anime_id_to_idx[aid])
        if pair not in interaction:
            interaction[pair] = IMPLICIT_SCORE

rows_idx, cols_idx, vals = [], [], []
for (r, c), v in interaction.items():
    rows_idx.append(r)
    cols_idx.append(c)
    vals.append(v)

user_anime_matrix = csr_matrix((vals, (rows_idx, cols_idx)), shape=(n_all_users, n_anime))
print(f"상호작용 행렬: {user_anime_matrix.shape}, nnz={user_anime_matrix.nnz}")

# 유저별 상호작용 수
user_interaction_counts = np.diff(user_anime_matrix.indptr)
print(f"평균 상호작용 수: {user_interaction_counts.mean():.1f}")
print(f"상호작용 0인 유저: {(user_interaction_counts == 0).sum()}")

상호작용 행렬: (184937, 8873), nnz=672882
평균 상호작용 수: 3.6
상호작용 0인 유저: 0


## 협업 필터링 (CF)

In [32]:
# 협업 필터링 (CF)
# 전체 유저 N×N 유사도 행렬은 메모리 초과 → 사이트 유저 행만 계산
# cosine_similarity(site_matrix, all_matrix) → (n_site, n_all) 크기
TOP_K_USERS = 20

site_matrix = user_anime_matrix[site_indices]  # (n_site, n_anime)
site_sim = cosine_similarity(site_matrix, user_anime_matrix)  # (n_site, n_all)
print(f"유사도 행렬: {site_sim.shape} ({site_sim.nbytes / 1024**2:.1f} MB)")

# 자기 자신 제외
for local_i, global_i in enumerate(site_indices):
    site_sim[local_i, global_i] = 0

# CF 점수 계산
watched_dense = user_anime_matrix.toarray()
watched_mask_site = {i: watched_dense[i] > 0 for i in site_indices}

cf_scores = np.zeros((len(site_indices), n_anime))

for local_i in range(len(site_indices)):
    sim_row = site_sim[local_i]
    top_k_idx = np.argsort(sim_row)[-TOP_K_USERS:]
    top_k_sims = sim_row[top_k_idx]

    if top_k_sims.sum() == 0:
        continue

    for j, s in zip(top_k_idx, top_k_sims):
        if s <= 0:
            continue
        neighbor_row = user_anime_matrix[j].toarray().flatten()
        cf_scores[local_i] += s * neighbor_row

    sim_sum = top_k_sims[top_k_sims > 0].sum()
    if sim_sum > 0:
        cf_scores[local_i] /= sim_sum

# 이미 본 애니 제외 + 0-1 정규화
for local_i, global_i in enumerate(site_indices):
    cf_scores[local_i][watched_mask_site[global_i]] = 0
    row_max = cf_scores[local_i].max()
    if row_max > 0:
        cf_scores[local_i] /= row_max

del site_sim  # 메모리 해제
print(f"CF 점수 계산 완료 (사이트 유저 {len(site_indices)}명)")

유사도 행렬: (2, 184937) (2.8 MB)
CF 점수 계산 완료 (사이트 유저 2명)


## 콘텐츠 기반 필터링 (CBF)

In [33]:
# Feature matrix (genres + tags + directors + companies)
mlb_genres = MultiLabelBinarizer()
mlb_tags = MultiLabelBinarizer()
mlb_directors = MultiLabelBinarizer()
mlb_companies = MultiLabelBinarizer()

vec_genres = mlb_genres.fit_transform(genres_list)
vec_tags = mlb_tags.fit_transform(tags_list)
vec_directors = mlb_directors.fit_transform(directors_list)
vec_companies = mlb_companies.fit_transform(companies_list)

feature_matrix = np.hstack([vec_genres, vec_tags, vec_directors, vec_companies]).astype(np.float32)
print(f"Feature matrix: {feature_matrix.shape}")

# 사이트 유저별 CBF 프로필 벡터 구성
POSITIVE_THRESHOLD = 6.0
n_features = feature_matrix.shape[1]

cbf_scores = np.zeros((len(site_indices), n_anime))

for local_i, global_i in enumerate(site_indices):
    scores = watched_dense[global_i]
    positive_mask = scores >= POSITIVE_THRESHOLD
    positive_indices = np.where(positive_mask)[0]

    profile = np.zeros(n_features, dtype=np.float32)
    if len(positive_indices) > 0:
        weights = scores[positive_indices]
        weighted_features = feature_matrix[positive_indices] * weights[:, np.newaxis]
        profile = weighted_features.sum(axis=0) / weights.sum()
    elif user_keys[global_i] in user_genre_map:
        # 콜드 스타트: 장르 선호도로 프로필 생성
        preferred_genres = user_genre_map[user_keys[global_i]]
        genre_classes = list(mlb_genres.classes_)
        for g in preferred_genres:
            if g in genre_classes:
                idx = genre_classes.index(g)
                profile[idx] = 1.0

    if profile.sum() > 0:
        sim = cosine_similarity(profile.reshape(1, -1), feature_matrix).flatten()
        sim[watched_mask_site[global_i]] = 0
        row_max = sim.max()
        if row_max > 0:
            sim /= row_max
        cbf_scores[local_i] = sim

has_profile = sum(1 for local_i in range(len(site_indices)) if cbf_scores[local_i].max() > 0)
print(f"CBF 프로필 있는 사이트 유저: {has_profile}/{n_site_users}")

Feature matrix: (8873, 2510)
CBF 프로필 있는 사이트 유저: 1/2


## 하이브리드 결합 + 결과 저장

CBF 점수 계산이 위 셀에 통합되었으므로 이 셀은 마크다운으로 전환합니다.

## 하이브리드 결합 + 결과 저장

In [34]:
# 하이브리드 결합 — 사이트 유저에 대해서만 추천 생성
# 동일 시리즈 중복 제거: series_mapping 그룹당 대표작 1개만 (최고 점수 우선)
TOP_N = 20
CANDIDATE_POOL = TOP_N * 3  # 중복 제거로 줄어들 것을 대비한 후보 풀
result_rows = []

for local_i, global_i in enumerate(site_indices):
    real_user_id = site_idx_to_user_id[global_i]
    count = user_interaction_counts[global_i]

    if count < 3:
        alpha = 0.0  # 순수 CBF
    elif count < 10:
        alpha = 0.3
    else:
        alpha = 0.6

    final = alpha * cf_scores[local_i] + (1 - alpha) * cbf_scores[local_i]

    # 충분한 후보를 확보하여 시리즈 중복 제거 후에도 TOP_N을 채울 수 있도록
    top_candidates = np.argsort(final)[-CANDIDATE_POOL:][::-1]

    # reason 결정
    if alpha == 0:
        reason = "GENRE" if count == 0 and user_keys[global_i] in user_genre_map else "CBF"
    elif alpha == 1.0:
        reason = "CF"
    else:
        reason = "HYBRID"

    seen_groups = set()
    picked = 0
    for idx in top_candidates:
        score = final[idx]
        if score <= 0:
            break

        aid = anime_ids[idx]
        group = anime_group_map.get(aid)
        if group is not None:
            if group in seen_groups:
                continue
            seen_groups.add(group)

        result_rows.append((
            real_user_id,
            aid,
            round(float(score), 4),
            reason,
        ))
        picked += 1
        if picked >= TOP_N:
            break

print(f"총 추천 레코드 수: {len(result_rows)}")
print(f"추천 대상 유저 수: {len(set(r[0] for r in result_rows))}")

총 추천 레코드 수: 20
추천 대상 유저 수: 1


In [35]:
# 테이블 생성 + 데이터 저장
cur.execute("""
    CREATE TABLE IF NOT EXISTS user_recommendations (
        id BIGSERIAL PRIMARY KEY,
        user_id BIGINT NOT NULL REFERENCES users(id),
        anime_id BIGINT NOT NULL REFERENCES anime(id),
        score NUMERIC(5, 4) NOT NULL,
        reason VARCHAR(10) NOT NULL,
        created_at TIMESTAMP NOT NULL DEFAULT NOW(),
        UNIQUE(user_id, anime_id)
    );
""")
pg.commit()

cur.execute("TRUNCATE TABLE user_recommendations RESTART IDENTITY;")

execute_batch(cur, """
    INSERT INTO user_recommendations (user_id, anime_id, score, reason)
    VALUES (%s, %s, %s, %s);
""", result_rows, page_size=1000)

pg.commit()
print(f"Inserted {len(result_rows)} rows into user_recommendations")

Inserted 20 rows into user_recommendations


## 검증

In [36]:
# 결과 확인
cur.execute("SELECT COUNT(*) FROM user_recommendations")
total = cur.fetchone()[0]
print(f"총 추천 레코드: {total}")

cur.execute("SELECT COUNT(DISTINCT user_id) FROM user_recommendations")
user_count = cur.fetchone()[0]
print(f"추천 대상 유저 수: {user_count}")

# reason 분포
cur.execute("SELECT reason, COUNT(*) FROM user_recommendations GROUP BY reason ORDER BY COUNT(*) DESC")
for reason, cnt in cur.fetchall():
    print(f"  {reason}: {cnt}")

# 샘플 출력
cur.execute("""
    SELECT r.user_id, u.nickname, a.name, r.score, r.reason
    FROM user_recommendations r
    JOIN users u ON r.user_id = u.id
    JOIN anime a ON r.anime_id = a.id
    ORDER BY r.user_id, r.score DESC
    LIMIT 15
""")
print("\n--- 샘플 추천 ---")
for row in cur.fetchall():
    print(f"  [{row[1]}] → {row[2]} (score={row[3]}, reason={row[4]})")

총 추천 레코드: 20
추천 대상 유저 수: 1
  HYBRID: 20

--- 샘플 추천 ---
  [이소영] → (더빙) SPY×FAMILY part 2 (score=0.7000, reason=HYBRID)
  [이소영] → 주술회전 2기 : 시부야 사변 (score=0.5666, reason=HYBRID)
  [이소영] → 옆자리 괴물군 (score=0.5487, reason=HYBRID)
  [이소영] → 앗군과 그녀 (score=0.5384, reason=HYBRID)
  [이소영] → (더빙) 너에게 닿기를 2기 - 판권 부활 (score=0.5331, reason=HYBRID)
  [이소영] → 호리미야 -piece- (score=0.5284, reason=HYBRID)
  [이소영] → 최근 고용한 메이드가 수상하다 (score=0.5129, reason=HYBRID)
  [이소영] → 러브 콤플렉스 (score=0.5129, reason=HYBRID)
  [이소영] → 아리스 or 아리스 (score=0.4922, reason=HYBRID)
  [이소영] → 어쨌든 귀여워 2기 (score=0.4860, reason=HYBRID)
  [이소영] → 꽃보다 남자 (score=0.4860, reason=HYBRID)
  [이소영] → 뻐꾸기 커플 시즌 2 (score=0.4663, reason=HYBRID)
  [이소영] → 귀엽기만 한 게 아닌 시키모리 양 (score=0.4663, reason=HYBRID)
  [이소영] → 초 GALS! 고토부키 란 (score=0.4653, reason=HYBRID)
  [이소영] → 고쿠센 (score=0.4653, reason=HYBRID)


In [37]:
cur.close()
pg.close()
print("Done!")

Done!
